# GEMS M9 — GeoFault production

**Physics-aware multiscale fault reconstruction and discovery system.** One coherent system, not a collection of experiments.

**Hypothesis (to be evaluated, never assumed).** A multiscale, physics-aware network trained to recognise and *reconstruct* incomplete fault systems, augmented by structured geological tracing and candidate-level verification, recovers fault geometry missed by conventional historical-label segmentation.

```text
RAW GEODAWN + CACHED GEOLOGICAL FEATURES (139 entries, reused)
                    |
        PHYSICS FEATURE PYRAMID (02)  +  FOURIER / WAVELET (03, gated)
                    |
   GeoFaultNet: modality stems -> gated fusion -> U-Net + low-res context (06)
       |-> dense fault evidence   |-> axial orientation   |-> endpoints / junctions
                    |
   CONDITIONAL RECONSTRUCTION training on damaged maps (05, 07, 08)
                    |
   STRUCTURED TRACING: crest+MCP baseline, beam search, branches (11)  [MC 12 / RL 13 optional]
                    |
   CANDIDATE FEATURES (14) -> XGBoost / LightGBM / CatBoost (15)
                    |
   DENSE + STRUCTURED (+ legacy B0/B1) CROSS-FITTED FUSION (16)  -> Gate 6
                    |
   FULL-AREA INFERENCE (17) -> VALIDATED FLOAT32 GEOTIFF: champion + B0/B1 fallback (18)
```

**Rules of this notebook**
* B0 / B1 stay immutable references and the submission fallback; M7 / M8 artifacts are read-only. M9 writes only under `outputs/m9_geofault_production/`.
* One protocol for every number: official kernel / α / β, known faults masked pixel-exactly, eligible-pixel budget, `gems3.official.FastScorer`. Every label-dependent choice for a zone is made on the other zones. These zones were inspected before: **not an untouched test set**.
* The dense channel never depends on candidate extraction: if tracing fails, the dense map is still a complete submission route.
* `PILOT` (default) trains one fold + one split with a short schedule; decisions are PROVISIONAL. `FULL` trains everything. Profiles change scope only.
* **No number exists until the cells run.** Markdown describes how to read outcomes; it never reports a measurement.

| section | subject | gate |
|---|---|---|
| 00 | configuration, environment, artifact recovery | stops on CRS / template mismatch |
| 01 | baseline recovery, metric verification, fallback GeoTIFF | **G0** |
| 02 | feature-store audit, geological representation | — |
| 03 | Fourier / wavelet pipeline, nested family screen | **G1** |
| 04 | training dataset and geographic CV (CV-A..F) | — |
| 05 | synthetic fault-map corruption | — |
| 06 | multiscale model architecture | — |
| 07 | multi-task losses | — |
| 08 | training and checkpoints (GPU) | — |
| 09 | geographic and discovery validation | **G2, G3** |
| 10 | dense structural prediction analysis | — |
| 11 | graph and beam-search fault tracing | **G4** |
| 12 | Monte Carlo uncertainty (optional) | G5 |
| 13 | optional RL tracing | G5 |
| 14 | candidate-level geological features | needs G4 |
| 15 | XGBoost / LightGBM / CatBoost | — |
| 16 | dense + structured fusion | **G6** |
| 17 | full-area inference | — |
| 18 | submission validation | never uploads |
| 19 | scientific reports and model registry | — |

Code: `gems_bench/m9/` · configuration: `configs/m9/geofault_production.yaml` · guide: `docs/M9_GEOFAULT_PRODUCTION.md`.

## 00. Configuration, environment and artifact recovery

* **Research objective.** Verify that the grid, template, folds, splits, cached features, B0 / B1 maps and earlier M7 / M8 artifacts are present and compatible; fix the configuration of this run.
* **Geological rationale.** Every later comparison is only meaningful on one grid (EPSG:32611, 100 m) and one fault-label definition.
* **Inputs.** Drive `GEMS_2/` (raw GeoTIFFs, feature store, folds, discovery splits, B0 / B1 experiment folders, `outputs/m7_discovery_first`, `outputs/m8_physics_probabilistic`).
* **Method.** Mount Drive, install the repository, open the pipeline context (data stages validated / staged, never rebuilt), create `m9 = pipe.m9(M9_CONFIG)`, run `s00_setup` (reuse matrix, band availability, protocol keys, M8 saved gate tables copied verbatim).
* **Expected artifacts.** `manifests/setup.json`, `manifests/reuse_matrix.csv`, `manifests/band_availability.csv`, `manifests/protocol_keys.csv`, `manifests/m9_config.json`, `baseline/m8_saved_outputs.csv` (+ `baseline/recovered_m8/*`).
* **Validation criteria.** A CRS or template mismatch raises. M8 tables are recorded as found ('absent' if M8 never ran): nothing about M8 is assumed.
* **Reload / resume.** Always re-run (read-only, seconds). After a disconnect: run the four setup cells, then any section.

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'gems3-discovery-research',               # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',                      # data stages reuse Drive; M9 trains its own network in section 08
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,                                 # section 08 / 17 need a GPU (T4 is enough); the rest runs on CPU
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
PRIVATE_REPO = False

M9_CONFIG = {
    'profile': 'PILOT',                              # PILOT | FULL (computational scope only)
    'profile_overrides': {},                         # e.g. {'epochs': 20}: scope keys only
    'overwrite_existing': False,                     # True recomputes stages (checkpoints are reused by spec hash)
    'sections': {                                    # independent switches (stages group them: PREPARE ... SUBMIT)
        's01_baseline': True, 's02_features': True, 's03_spectral': True, 's04_dataset': True,
        's05_corruption': True, 's06_architecture': True, 's07_losses': True,
        's08_training': True,                        # GPU; resumable from checkpoints
        's09_validation': True, 's10_dense_analysis': True, 's11_tracing': True,
        's12_monte_carlo': False,                    # P2: only after Gate 4 / when its value is to be tested
        's13_rl': False,                             # P2: optional challenger
        's14_candidate_features': True, 's15_gbm_benchmark': True, 's16_fusion': True,
        's17_inference': False,                      # GPU recommended
        's18_submission': False,                     # writes validated GeoTIFFs; never uploads
        's19_reports': True,
    },
    'shared': {                                      # M7-schema protocol (units, budget, promotion rules)
        'models': ['b0_reference', 'b1_bg_weight'],
        'submission': {'source': 'oof_stitched', 'area_frac_source': 'geo'},   # 'fold_average' needs a GPU
    },
    'training': {'max_runtime_min': 0},              # e.g. 200 on a free session: clean, resumable stop
    'tracing': {'force_rank': False},
}
SHOW_FIGURES = True

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab,m9]'], check=True)
if M9_CONFIG['sections'].get('s13_rl'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[rl]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.pipeline import Pipeline

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m9 = pipe.m9(M9_CONFIG)
OUT = m9.ws.root
print('profile:', m9.profile)
print('M9 outputs (local):', OUT)
print('M9 outputs (Drive):', m9.cfg['_drive_root'])

def show(names, sub):
    if not (SHOW_FIGURES and m9.cfg.get('display_figures', True)):
        return
    for n in names:
        p = OUT / sub / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', sub, n)

def tab(name, sub, n=30):
    t = m9.ws.load_table(name, sub)
    display(t.head(n)) if len(t) else print('empty / not produced:', sub, name)
    return t

In [ ]:
setup = m9.s00_setup()                    # read-only; raises if CRS / template checks fail
print(json.dumps({k: v for k, v in setup['checks'].items() if k != 'bands'}, indent=1, default=str))
display(setup['bands'])
display(setup['reuse'])                   # static reuse matrix: capability -> existing module -> reuse / change -> consumer
display(setup['m8'])                      # M8 saved outputs as found on Drive (nothing assumed)
display(m9.m7.protocol_keys())

## 01. Historical baseline recovery and metric verification — Gate 0

* **Research objective.** Re-establish the B0 / B1 champion (`fusion_thin` = rank fusion + crest thinning) under the shared official-style protocol and prove that a template-compliant submission can be produced before anything new is built.
* **Geological rationale.** A new representation only matters if it beats the strongest existing model on identical units, masks and budget.
* **Inputs.** B0 / B1 OOF + discovery maps (M6 inventory), M7 `baseline_selection.json` when its protocol keys still match.
* **Method.** M8's baseline recovery (`m8.baseline.run`, unchanged): reuse M7's nested selection when protocol keys match, otherwise recompute with M7's code; rebuild every per-zone champion map (`validation/maps/s01_*`); compatibility checks; recorded offline values next to recomputed ones (never reconciled). Gate 0: write and validate the fallback GeoTIFF (`m7.export.export_baseline`, ring 0).
* **Expected artifacts.** `baseline/baseline_table__<fp>.csv` (immutable), `recorded_vs_this_run.csv`, `compatibility.csv`, `system_table.csv`, `baseline_selection.json`, `gate0.csv`, `submissions/m9_fallback_fusion_thin.tif` + `.json`.
* **Validation criteria.** No compatibility row `DIFFERS`; Gate 0 `PASS` (single float32 band, EPSG:32611, template grid, [0, 1], NaN only where the template is NaN).
* **Reload / resume.** Fingerprinted; re-running reuses everything. The fallback file is never overwritten (a timestamped copy is written instead).

In [ ]:
if m9.enabled('s01_baseline'):
    s01 = m9.s01_baseline()
    t = s01['table']
    display(t[[c for c in ('task', 'method', 'dti_crossfit', 'fold_min', 'optimism', 'area_frac_eligible_mean') if c in t]])
    display(s01['recorded'])
    comp = s01['compatibility']
    display(comp)
    if len(comp) and (comp.status == 'DIFFERS').any():
        print('WARNING: compatibility differences:', comp[comp.status == 'DIFFERS'].check.tolist())
    tab('gate0', 'baseline')
    show(['baseline_dti'], 'baseline')

## 02. Feature-store audit and geological representation

* **Research objective.** Turn the ~139 cached features into a compact, non-redundant, provenance-tracked input pyramid; add only the cross-modal evidence that does not exist yet.
* **Geological rationale.** Potential-field gradients, tilt / normalised derivatives, curvature, ridges, directional filters, coherence and terrain morphology express faults differently. Magnetic and gravity maxima need not coincide (different depths, dip, ~2 km gravity resolution), so agreement must be spatially tolerant and never required. Strain / seismicity are regional descriptors, not fault locators: they feed the low-resolution branch only.
* **Inputs.** Feature store (Phase 2 / EDA / Phase 4 / Phase 5), raw bands.
* **Method.** Inventory with family rules; greedy label-free Spearman pruning per family (B0's channels kept first); new layers: tolerant magnetic–gravity alignment (300 / 800 m), tolerant terrain–magnetic alignment (surface / subsurface), tolerant orientation agreement, relative magnetic / gravity strength, localized contrast (LCN). No label is read.
* **Expected artifacts.** `features/inventory.csv` (family, units, resolution, cache state, fingerprint), `selected_registry.csv`, `redundancy_pairs.csv`, `crossmodal_new.csv`, `selected_spearman.csv`, `features/figures/feature_families.png`; layers in `cache/m9/layers`.
* **Validation criteria.** Every selected channel has provenance and a fingerprint; no family exceeds its cap; supervised value is judged only in section 03 (nested).
* **Reload / resume.** Fingerprinted; cached layers are reused (rebuilt only if the cache is lost).

In [ ]:
if m9.enabled('s02_features'):
    s02 = m9.s02_features()
    reg = s02['registry']
    display(reg.groupby(['family', 'selected']).size().rename('n').unstack(fill_value=0))
    tab('crossmodal_new', 'features')
    tab('redundancy_pairs', 'features', 20)
    show(['feature_families'], 'features/figures')

## 03. Fourier / wavelet feature pipeline — Gate 1

* **Research objective.** Extract structural frequency, orientation and multiscale information not already represented, and decide (nested) whether it — and the new cross-modal layers — earn a place in the production inputs.
* **Geological rationale.** A linear structure concentrates spectral energy along the wavevector *perpendicular* to its trace (trace strike = wavevector azimuth + 90°). Anisotropy, directional coherence and scale persistence describe fabric that single-scale gradients miss. Upward continuation is the stable physical scale-space of a potential field; downward continuation is unstable and is not used. Spectral slopes are not interpreted as depths.
* **Inputs.** Magnetic (`rtp`) and gravity (`iso_grav_anom`) fields.
* **Method.** Overlapping 6.4 km windows (Hann taper, plane removal, nodata-aware) → anisotropy, dominant strike (doubled angle), directional entropy, short / long energy ratio, radial slope, band energy; steerable one-sided log-Gabor wavelets at 0.4–3.2 km → energy per scale, orientation, coherence, persistence, fine / coarse ratio; continuation bands; optional STFT ablation. Label-free pruning and comparison with existing Gabor / ridge / DoG features. **Gate 1** evidence: pixel LightGBM screen trained on each trained zone's TRAINING geography only, scored on its held-out zone with the official scorer (`raw`, `base`, `base_no_new`, `base_spectral`), unique systems, nested consistency.
* **Expected artifacts.** `spectral/spectral_registry.csv`, `spectral_vs_existing.csv`, `fft_windows_<field>.npz`, `screen_per_zone.csv`, `screen_pooled.csv`, `screen_unique_systems.csv`, `screen_nested_consistency.csv`, `gate1.csv`; figures `fourier_*`, `wavelet_*`, `mag_grav_agreement`, `feature_set_comparison`, `raw_vs_structural`.
* **Validation criteria.** Gate 1 INCLUDE a component when its pooled DTI gain ≥ 0.003 or it recovers ≥ 2 systems the base set misses; otherwise EXCLUDE (kept for diagnostics). The `no_spectral` network variant (FULL) confirms the decision at network level (section 09 `variant_deltas`).
* **Reload / resume.** Layers cached in `cache/m9/layers`; the stage is fingerprinted.

In [ ]:
if m9.enabled('s03_spectral'):
    s03 = m9.s03_spectral()
    sub = 'spectral'
    tab('spectral_registry', sub, 40)
    sv = m9.ws.load_table('spectral_vs_existing', sub)
    if len(sv):
        display(sv.sort_values('abs_spearman', ascending=False).head(20))
    tab('screen_pooled', sub)
    tab('screen_unique_systems', sub)
    tab('gate1', sub)
    show(['feature_set_comparison', 'raw_vs_structural', 'mag_grav_agreement', 'fourier_magnetic', 'fourier_gravity',
          'wavelet_magnetic', 'wavelet_gravity'], f'{sub}/figures')

## 04. Training dataset and geographic CV

* **Research objective.** Build the input cube once, define training units, fit per-unit robust scaling on permitted geography only, and write the CV-A..F manifests.
* **Geological rationale.** Faults are spatially autocorrelated; geographic separation (buffered folds) and whole-system withholding are the only honest tests of generalisation and discovery.
* **Inputs.** Selected channels (02, 03), folds, discovery splits.
* **Method.** `geo_f<k>` (CV-A: inner-train patches, inner-val checkpointing, predict the validation zone), `disc_s<d>` (CV-B: withheld systems + ring weight 0, never negatives, never shown as context), optional `full`. Cube: float16 memory-mapped local channels + footprint, average-pooled cube for the low-resolution branch. Scaler: median / IQR on a random sample of the unit's permitted pixels. CV-D length strata, CV-E label-free geological domains (k-means of 20 km block descriptors), CV-F distance bands to known faults.
* **Expected artifacts.** `datasets/training_units.csv`, `normalization_manifest.json`, `channels_<variant>.json`, `datasets/cv/cv_a_geographic.csv`, `cv_b_discovery.csv`, `cv_d_length_strata.csv`, `cv_f_proximity.csv`, `domains.tif` / `.npz`, `cv_e_domain_blocks.csv`; cube in `cache/m9`.
* **Validation criteria.** Every scaler hash recorded; trained zones match the profile; domains are label-free.
* **Reload / resume.** The cube is rebuilt only when its channel list changes (key in the file name).

In [ ]:
if m9.enabled('s04_dataset'):
    s04 = m9.s04_dataset()
    display(s04['units'])
    for n in ('cv_a_geographic', 'cv_b_discovery', 'cv_d_length_strata', 'cv_f_proximity'):
        tab(n, 'datasets/cv')

## 05. Synthetic fault-map corruption

* **Research objective.** Teach the network to reconstruct incomplete maps (truncated endpoints, interior gaps, omitted segments / branches, displaced traces, removed systems) while still discovering faults with no map at all.
* **Geological rationale.** Mapping stops where access or exposure stops, not where faults stop. The test-time known map is incomplete in exactly these ways, and new faults may be continuations or corrections next to it.
* **Inputs.** Visible labels of each training unit, the discovery system map.
* **Method.** Mixture none / partial / heavy (35 / 40 / 25 %); operators applied per patch to visible labels only; the context channel shows the damaged map plus a flag (1 = a map is given); seg targets stay the full visible map, but pixels shown as context get weight 0.1 (copying earns little); removed geometry drives L6. CV-C benchmark sets: M7 corruption classes on each trained fold, systems assigned to folds before damage.
* **Expected artifacts.** `corruption/samples/<unit>.npz`, `samples_<unit>.csv`, `corruption_samples.png`, `operation_counts.csv`, `corruption_manifest.json`, `corruption/benchmark/fold<k>.npz`, `benchmark_protocol.csv`.
* **Validation criteria.** Leakage audit: without displacement, the context never shows a pixel that is not a visible label (raises otherwise).
* **Reload / resume.** Fingerprinted; benchmark sets are seeded.

In [ ]:
if m9.enabled('s05_corruption'):
    s05 = m9.s05_corruption()
    tab('operation_counts', 'corruption')
    tab('benchmark_protocol', 'corruption')
    for p in sorted((OUT / 'corruption' / 'samples').glob('*.png')):
        display(Image(filename=str(p)))

## 06. Multiscale model architecture

* **Research objective.** One practical multi-resolution, multi-task network family (GeoFaultNet); no architecture zoo.
* **Geological rationale.** Fault localisation is local (100 m), its plausibility regional (km). Modalities must keep separate evidence (no forced agreement); orientation is axial (θ ≡ θ + 180°).
* **Inputs.** `channels_<variant>.json`, model block of the configuration.
* **Method.** Modality stems (raw, structural, spectral, optional DEM, map context) → per-pixel gated fusion → residual U-Net (P … P/16) with a 4× wider low-resolution context branch aligned at the bottleneck + FiLM → heads: dense evidence, (cos 2θ, sin 2θ), endpoint, junction.
* **Expected artifacts.** `models/architecture.csv` (parameters, channels per stem), `models/net_spec_<variant>.json`.
* **Validation criteria.** Builds on CPU; parameter count stays in the few-million range (T4-friendly).
* **Reload / resume.** Cheap; fingerprinted.

In [ ]:
if m9.enabled('s06_architecture'):
    s06 = m9.s06_architecture()
    display(s06['architecture'])

## 07. Multi-task losses

* **Research objective.** A staged, incomplete-label-aware objective whose terms are switched on in a fixed curriculum, never tuned jointly.
* **Geological rationale.** Unmapped faults exist: unlabeled background is a weak negative. Thin, continuous, oriented traces need connectivity and orientation supervision; endpoints / junctions carry network topology.
* **Inputs.** Loss block, variants.
* **Method.** L1 focal PU-weighted BCE; L2 kernel Tversky surrogate (α 0.2, β 0.8, 300 m) — a surrogate, the exact scorer evaluates; L3 skeleton recall / clDice; L4 doubled-angle cosine; L5 sparse topology heatmaps; L6 reconstruction; L7 optional EMA consistency. Curriculum: A (L1 + L6), B (+ L4 + L5), C (+ enabled Stage-C terms).
* **Expected artifacts.** `training/loss_schedule.csv`.
* **Validation criteria.** Stage-C terms enter production only through the `stage_c` variant after it beats `core` (section 09 `variant_deltas`).
* **Reload / resume.** Cheap; fingerprinted.

In [ ]:
if m9.enabled('s07_losses'):
    display(m9.s07_losses()['schedule'])

## 08. Training and checkpoints (GPU)

* **Research objective.** Train every (variant, unit) of the profile reproducibly and resumably.
* **Geological rationale.** Discovery units withhold whole systems; their epochs follow the geo units' median best epoch (no peeking at the withheld truth).
* **Inputs.** Cube, targets, corruption, losses.
* **Method.** AMP (bf16 on Ampere+, fp16 + scaler on T4), gradient accumulation, clipping, OneCycle, CPU patch prefetch (the full cube never goes to VRAM), inner-validation checkpoint score = official-style DTI with no map context, patience, collapse guard, OOM back-off, `best_and_latest` atomic checkpoints synced to Drive, `max_runtime_min` clean stop.
* **Expected artifacts.** `models/checkpoints/<variant>/<unit>_model.pt` (+ resume files while running), `models/<variant>__<unit>_scaler.json`, `training/logs/<variant>__<unit>.csv` (loss decomposition per epoch), `training/training_units.csv`, `reports/model_registry/model_registry.csv` (sha256), `training/loss_components.png`.
* **Validation criteria.** Status `complete` / `early_stopped`; `stopped_collapse` means the PU weighting is too lax for that unit (inspect `loss_unlabeled_mean_p`).
* **Reload / resume.** Re-run the cell: finished units are reused by spec hash, interrupted ones resume from the last epoch.

In [ ]:
if m9.enabled('s08_training'):
    s08 = m9.s08_training()               # None when interrupted by the runtime budget: re-run to resume
    tab('training_units', 'training')
    show(['loss_components'], 'training')

## 09. Geographic and discovery validation — Gates 2 and 3

* **Research objective.** Measure the dense channel under CV-A..F against the champion on identical units, and decide whether the redesign (Gate 2) and the reconstruction task (Gate 3) earn their place.
* **Geological rationale.** Geographic transfer, whole-system discovery, short faults, domain transfer and proximity to known faults fail for different reasons; one aggregate hides them.
* **Inputs.** Checkpoints, CV-C sets, champion maps.
* **Method.** Each unit predicts only its own held-out mask (geo: no context; disc: no context AND known retained map). Nested (rep, frac) per zone; PILOT borrows the champion's choice (labelled). M7's comparison re-scores every map: pooled + worst-zone DTI, system / length / short-fault recall, density, regions, known-fault proximity, paired zone deltas, spatial block bootstrap. CV-C reconstruction recall with damaged-map vs no context; CV-D length strata; CV-E domain DTI; variant deltas.
* **Expected artifacts.** `predictions/<variant>/*.npz`, `predictions/oof/<variant>_seg.tif`, `predictions/discovery/<variant>_seg.tif`, `validation/maps/m9_dense_*`, `validation/metrics/{method_pooled, method_per_zone, paired_zone_deltas, bootstrap, system_hits, known_proximity, regional_dti, reconstruction(_pooled), length_strata, domain_dti, variant_deltas}`, `reports/decision_verdicts.csv`, `validation/gate2.csv`, `gate3.csv`, figures in `validation/figures/`.
* **Validation criteria.** Gate 2 PROMOTE (all promotion rules) or COMPLEMENT (≥ 3 unique systems, or a fusion gain ≥ 0.005 in section 16); Gate 3 RETAIN_RECONSTRUCTION when reconstruction recall rises ≥ 0.02 without losing ≥ 0.005 discovery / geographic DTI (needs `recon_off`, FULL).
* **Reload / resume.** Predictions are reused file by file; the stage is fingerprinted.

In [ ]:
if m9.enabled('s09_validation'):
    s09 = m9.s09_validation()
    sub = 'validation/metrics'
    tab('method_pooled', sub, 40)
    pz = m9.ws.load_table('paired_zone_deltas', sub)
    if len(pz):
        display(pz.pivot_table(index=['task', 'method'], columns='zone', values='delta_vs_ref'))
    tab('bootstrap', sub)
    tab('reconstruction_pooled', sub, 40)
    tab('variant_deltas', sub)
    tab('gate2', 'validation')
    tab('gate3', 'validation')
    show(['geographic_fold_performance', 'whole_system_recall', 'short_fault_behaviour', 'known_fault_distance',
          'geographic_failure_modes'], 'validation/figures')

## 10. Dense structural prediction analysis

* **Research objective.** Check the auxiliary heads: is the predicted orientation right where it is confident; are endpoints and junctions found; how does the score relate to proximity of truth?
* **Geological rationale.** Tracing (11) relies on the orientation field; topology heads justify branch / stop decisions.
* **Inputs.** Held-out `core` predictions.
* **Method.** Axial error vs label strike by confidence bin; endpoint / junction peak recall and precision within 2 px; reliability (share within 300 m of truth by score bin — not calibration: unlabelled faults count as misses).
* **Expected artifacts.** `dense_analysis/orientation_error.csv`, `endpoint_junction.csv`, `reliability.csv`, `reliability.png`.
* **Validation criteria.** Orientation error should fall with confidence; if not, tracing must lean on evidence, not orientation (`w_orient`).
* **Reload / resume.** Fingerprinted.

In [ ]:
if m9.enabled('s10_dense_analysis'):
    s10 = m9.s10_dense_analysis()
    for n in ('orientation_error', 'endpoint_junction', 'reliability'):
        tab(n, 'dense_analysis')
    show(['reliability'], 'dense_analysis')

## 11. Graph and beam-search fault tracing — Gate 4

* **Research objective.** Turn dense evidence + orientation into explicit fault hypotheses (with branches and alternatives) and measure, before any ranker, whether they cover systems the dense map misses.
* **Geological rationale.** Faults are continuous, gently curving, may be locally invisible (gaps) and branch; a path search with orientation-aware costs can bridge weak evidence that pixel thresholds drop.
* **Inputs.** Held-out dense predictions of each trained unit.
* **Method.** Baseline: crest skeleton + minimum-cost bridges. Beam search (vectorised over seeds, both directions, controlled turning, gap tolerance, branch re-tracing), optional stochastic beams; dedup by overlap; nothing is permanently thresholded away (statistics kept for ranking). Oracle: system recall vs density, additional systems over the dense map at the same pixel count.
* **Expected artifacts.** `tracing/units/<unit>.{npz,csv}`, `tracing/candidates.gpkg`, `tracer_log.csv`, `tracing/oracle/{oracle_curve, pooled_oracle, systems_at_max_density, gate4}.csv`, `oracle_curve.png`, `tracing_failures_*.png`.
* **Validation criteria.** Gate 4 CONTINUE when system recall at 40 km / 100 km² ≥ 0.30 and ≥ 3 additional systems; STOP → sections 14–15 refuse (`tracing.force_rank` overrides) and the dense channel remains the production route.
* **Reload / resume.** Trace sets are saved per unit; fingerprinted.

In [ ]:
if m9.enabled('s11_tracing'):
    s11 = m9.s11_tracing()
    tab('tracer_log', 'tracing')
    tab('pooled_oracle', 'tracing/oracle')
    tab('gate4', 'tracing/oracle')
    show(['oracle_curve'], 'tracing/oracle')
    for p in sorted((OUT / 'tracing').glob('tracing_failures_*.png'))[:3]:
        display(Image(filename=str(p)))

## 12. Monte Carlo uncertainty *(P2, off by default)*

* **Research objective.** Quantify which traced structures are stable under plausible evidence / processing perturbations (MC-A) and which alternatives a stochastic tracer proposes (MC-B); MC-C = seeded ensembles (`n_seeds`).
* **Geological rationale.** Edge positions depend on noise, registration, smoothing and thresholds; real structures should survive moderate perturbation.
* **Inputs.** Held-out dense predictions, section-11 traces.
* **Method.** 16 realisations (PILOT): correlated noise, one registration shift per realisation (shared source perturbed once), smoothing / orientation / seed jitter → re-trace; stochastic beams → occupancy, branch frequency; convergence check (raise realisations only if not converged AND Gate 5 shows value).
* **Expected artifacts.** `monte_carlo/stability.csv`, `convergence.csv`, `realization_params.csv`, `occupancy_<unit>.{npz,tif}`, `mc_only_coverage.csv`, `assumptions.json`, `mc_<unit>.png`, `reports/gate5.csv` (monte_carlo row).
* **Validation criteria.** Gate 5 INCLUDE only with ≥ 2 MC-only systems or a ranking gain ≥ 0.003 (`all_mc` vs `all`, section 16). Occupancy is never called a probability.
* **Reload / resume.** Fingerprinted.

In [ ]:
if m9.enabled('s12_monte_carlo'):
    s12 = m9.s12_monte_carlo()
    st = m9.ws.load_table('stability', 'monte_carlo')
    if len(st):
        display(st.describe().T)
    tab('convergence', 'monte_carlo')
    tab('mc_only_coverage', 'monte_carlo')
    tab('gate5', 'reports')
    for p in sorted((OUT / 'monte_carlo').glob('mc_*.png'))[:2]:
        display(Image(filename=str(p)))

## 13. Optional RL tracing *(P2, challenger only)*

* **Research objective.** Test whether a learned tracer beats beam search on held-out systems.
* **Geological rationale.** Experts trace faults by local decisions (continue, turn, jump a gap, branch, stop); a policy might learn those decisions from mapped traces.
* **Inputs.** Held-out dense predictions; truth of OTHER zones only (2 km buffer around the evaluated unit).
* **Method.** Gymnasium-compatible environment (advance / turn / connect / branch / stop); behavioural cloning on greedy expert moves; PPO / DQN via stable-baselines3 when installed (`rl.algorithm`); learned edge cost (logistic regression) used inside beam search; all scored with the same seeds and oracle as beam search.
* **Expected artifacts.** `rl/tracer_comparison.csv`, `reports/gate5.csv` (rl row).
* **Validation criteria.** Promote a challenger only if it recovers ≥ 2 more held-out systems than beam search at the gate density; otherwise beam search stays.
* **Reload / resume.** Fingerprinted; never required by the production path.

In [ ]:
if m9.enabled('s13_rl'):
    s13 = m9.s13_rl()
    tab('tracer_comparison', 'rl', 40)
    tab('gate5', 'reports')

## 14. Candidate-level geological features *(needs Gate 4 CONTINUE)*

* **Research objective.** Describe each traced hypothesis with interpretable, provenance-tracked evidence.
* **Geological rationale.** A fault juxtaposing contrasting units shows a localised fault-normal step; lithological contacts and basin edges also produce edges, so cross-modal consistency, spectral fabric and network context help separate them.
* **Inputs.** Trace sets, dense / legacy predictions, physics and spectral layers, optional Monte Carlo stability.
* **Method.** M7 descriptors + tracing statistics, fault-normal contrasts at 200 / 400 / 800 m (magnetic, gravity, conductivity, basement), spectral / cross-modal means, M9 and B0 / B1 support and disagreement, context percentiles, Monte Carlo stability (`has_mc` explicit). Labels: 1 near scored truth, 0 far, −1 unlabelled (dropped, never negative). IDs / zones / seed positions excluded from features.
* **Expected artifacts.** `candidates/candidate_table.parquet`, `candidates/units/*`, `feature_groups.csv`, `feature_provenance.csv`, `label_balance.csv`, `candidates.gpkg`, `candidate_fault_normal_profiles.png`.
* **Validation criteria.** Every feature belongs to a group; label balance is reported per task.
* **Reload / resume.** Fingerprinted.

In [ ]:
if m9.enabled('s14_candidate_features'):
    s14 = m9.s14_candidate_features()     # None (logged) when Gate 4 is STOP
    tab('label_balance', 'candidates')
    fg = m9.ws.load_table('feature_groups', 'candidates')
    if len(fg):
        display(fg.groupby('group').size())
    show(['candidate_fault_normal_profiles'], 'candidates')

## 15. XGBoost / LightGBM / CatBoost

* **Research objective.** A fair three-library comparison; choose the model with the best deployable geometry, preferring steadiness over a marginal aggregate gain.
* **Geological rationale.** The structured channel only matters if verified candidates add systems the dense map misses at a controlled density.
* **Inputs.** Candidate table, trace sets.
* **Method.** Identical tables, folds (leave-one-zone-out, 2 km buffer, linked-network exclusion), supervision, weights, seeds and budgets; candidate AUC-PR / ROC-AUC (reported, not optimised), top-10 % recall, system recall at the gate density, false-candidate share, rasterised official-style DTI with nested frac, seed stability; optional LambdaMART.
* **Expected artifacts.** `gbm/{summary, per_zone, candidate_metrics, system_recall, audit, availability, stability, choice, feature_importance}.csv`, `gbm/models/<library>_all.joblib`, `validation/maps/m9_struct_*`, `validation/maps/m9_structured`, `library_comparison.png`.
* **Validation criteria.** Choice = best pooled DTI; libraries within 0.003 compete on their worst zone.
* **Reload / resume.** Fingerprinted.

In [ ]:
if m9.enabled('s15_gbm_benchmark'):
    s15 = m9.s15_gbm_benchmark()
    tab('availability', 'gbm')
    tab('summary', 'gbm', 40)
    tab('choice', 'gbm')
    tab('stability', 'gbm')
    show(['library_comparison'], 'gbm')

## 16. Dense + structured fusion — Gate 6

* **Research objective.** Decide, leakage-safely, whether any M9 system replaces the B0 / B1 champion.
* **Geological rationale.** Dense evidence keeps faint, unlinked signal; structured traces keep continuity; legacy maps carry what B0 / B1 already learned. Ranking must not erase faint dense evidence.
* **Inputs.** Held-out dense maps, structured scores, champion maps.
* **Method.** Per-zone percentile channels D / S / L; fusions D+S, D+L, D+S+L on a weight simplex; weights, representation and budget nested on the other zones; the 300 m ring evaluated separately (never in the champion by default); M7 comparison + verdicts; Gate 6 promotes only a reproducible method that passes every rule. Gate 2 is updated with the fusion gain, Gate 5 with the Monte Carlo ranking gain.
* **Expected artifacts.** `fusion/{grid, summary, per_zone, ring, gate6}.csv`, `fusion/method_parameters.json`, `validation/metrics/*` (final), `reports/decision_verdicts.csv`, `dense_vs_structured.png`.
* **Validation criteria.** PROMOTE only with pooled gain ≥ 0.01, ≥ 2/3 zone wins, worst zone ≥ −0.01, geographic drop ≤ 0.01, bootstrap P(Δ > 0) ≥ 0.8, short-fault recall drop ≤ 0.02 — and ≥ 3 zones (PILOT can never promote).
* **Reload / resume.** Fingerprinted.

In [ ]:
if m9.enabled('s16_fusion'):
    s16 = m9.s16_fusion()
    tab('summary', 'fusion', 40)
    tab('ring', 'fusion')
    tab('method_pooled', 'validation/metrics', 60)
    tab('decision_verdicts', 'reports', 40)
    tab('gate6', 'fusion')
    show(['dense_vs_structured'], 'fusion')

## 17. Full-area inference *(GPU recommended)*

* **Research objective.** Produce the full-grid dense heads (and optional full-region traces) for submission.
* **Geological rationale.** At test time the whole known map is given: the production context is the known map (configurable).
* **Inputs.** `core` checkpoints (fold average or `full`), cube, chosen ranker.
* **Method.** 1024 px tiles with a 64 px halo, Hann-blended patches, linear seam ramps, resumable per tile, CPU / GPU fallback; full-region traces scored by the chosen library refitted on every labelled primary-task row.
* **Expected artifacts.** `predictions/full/core_heads.tif` (seg, cos 2θ, sin 2θ, endpoint, junction), `inference/tiles/*`, `tracing/full_region.gpkg`, `inference/inference_manifest.json`.
* **Validation criteria.** No NaN inside the footprint; tiles complete.
* **Reload / resume.** Re-run: finished tiles are skipped.

In [ ]:
if m9.enabled('s17_inference'):
    s17 = m9.s17_inference()
    print(json.dumps(m9.read_json('inference/inference_manifest.json'), indent=1, default=str))

## 18. Submission validation *(never uploads)*

* **Research objective.** Champion and fallback GeoTIFFs that pass every check.
* **Geological rationale.** —
* **Inputs.** Gate 6, method parameters, full-area products, champion member maps.
* **Method.** Fallback = B0 / B1 champion (ring 0). M9 champion only on Gate 6 PROMOTE: full-region score, all-zone (rep, frac) with the frac from the geo selection, eligible budget; shared writer (template authority, NaN only where the template is NaN, float32 in [0, 1], area ledger, never overwrite) + M9 checks (positive-area fraction within bounds, no known-fault positives, checkpoint sha256 vs registry).
* **Expected artifacts.** `submissions/*.tif` + `.json` manifests, `submissions/submission_index.csv`.
* **Validation criteria.** `status = VALID` for the file you upload; upload at most one controlled probe and log the score manually.
* **Reload / resume.** Never overwrites (timestamped copies).

In [ ]:
if m9.enabled('s18_submission'):
    s18 = m9.s18_submission()
    tab('submission_index', 'submissions')
    for f in sorted((OUT / 'submissions').glob('*.json')):
        j = json.loads(f.read_text())
        if 'area_ledger' in j:
            print(f.name)
            display(pd.DataFrame(j['area_ledger']))

## 19. Scientific reports and model registry

* **Research objective.** One readable account of what was tested, what passed and what stays.
* **Geological rationale.** —
* **Inputs.** Every saved table.
* **Method.** Gate collection with meaning and next action, report rebuilt from saved tables only (PILOT decisions PROVISIONAL), model registry (every checkpoint with sha256, cube key, config fingerprint; every submission file).
* **Expected artifacts.** `reports/gates.csv` / `.json`, `reports/M9_GEOFAULT_PRODUCTION_REPORT.md`, `reports/model_registry/model_registry.{csv,json}`.
* **Validation criteria.** No number in the report that is not in a saved table.
* **Reload / resume.** Always re-run (cheap).

In [ ]:
if m9.enabled('s19_reports'):
    s19 = m9.s19_reports()
    display(s19['gates'][['gate', 'component', 'decision', 'meaning', 'next_action']])
    display(Markdown((OUT / 'reports' / 'M9_GEOFAULT_PRODUCTION_REPORT.md').read_text()[:6000]))

## Run log, stages and resume

`m9.run(stage='PREPARE')`, `'TRAIN'`, `'VALIDATE'`, `'TRACE'`, `'RANK'`, `'INFER'`, `'SUBMIT'` run one stage group (enabled sections only); `m9.run()` runs everything. Finished sections are reused by fingerprint (`manifests/units/`), training resumes from checkpoints, inference from tiles. To scale up: `M9_CONFIG['profile'] = 'FULL'` (all folds / splits, all variants). Read first: `baseline/compatibility.csv`, `spectral/gate1.csv`, `validation/gate2.csv`, `tracing/oracle/gate4.csv`, `fusion/gate6.csv`, `reports/M9_GEOFAULT_PRODUCTION_REPORT.md`.

In [ ]:
display(pd.DataFrame(m9.log)) if m9.log else print('no stage ran in this session')
display(m9.summary())
m9.paths()
pipe.push_all()